# Regression Model (Ratings Prediction)

This notebook implements a ridge regression-based recommender that predicts user star ratings for restaurants.

### Features:
- Uses TF-IDF on review text

Metadata:
- business_avg_stars
- business_review_count
- latitude / longitude
- num_categories
- num_attributes
- city/state
- is_open

In [21]:
import ast
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder


## 1) Load Data

In [22]:
from pathlib import Path

# Dataset directory relative to the repository root / notebook location.
DATA_DIR = Path("team_project_dataset")

restaurants_path = DATA_DIR / "restaurants_santa_barbara.csv"
train_reviews_path = DATA_DIR / "train_reviews_santa_barbara.csv"
test_reviews_path = DATA_DIR / "test_reviews_santa_barbara.csv"

df_restaurants = pd.read_csv(restaurants_path)
df_train_reviews = pd.read_csv(train_reviews_path)
df_test_reviews = pd.read_csv(test_reviews_path)

df_restaurants.head()


,business_id,name,address,city,state,postal_code,categories,latitude,longitude,attributes
0,IDtLPgUrqorrpqSLdfMhZQ,Helena Avenue Bakery,"131 Anacapa St, Ste C",Santa Barbara,CA,93101,"food, restaurants, salad, coffee & tea, breakf...",34.414445,-119.690672,"{'RestaurantsTakeOut': 'True', 'NoiseLevel': ""..."
1,SZU9c8V2GuREDN5KgyHFJw,Santa Barbara Shellfish Company,230 Stearns Wharf,Santa Barbara,CA,93101,"live/raw food, restaurants, seafood, beer bar,...",34.408715,-119.685019,"{'OutdoorSeating': 'True', 'RestaurantsAttire'..."
2,ifjluUv4VASwmFqEp8cWlQ,Marty's Pizza,2733 De La Vina St,Santa Barbara,CA,93105,"pizza, restaurants",34.436236,-119.726147,"{'Alcohol': ""u'none'"", 'BusinessAcceptsCreditC..."
3,UFpCraqzFBAhtZqmxmiWsA,Cat Therapy,"1213 State St, Ste L",Santa Barbara,CA,93101,"themed cafes, cafes, pets, arts & entertainmen...",34.423302,-119.705471,"{'WheelchairAccessible': 'True', 'WiFi': ""u'fr..."
4,Hqz96v1ymucUKNzIWfEKXw,Subway,"1936 State St, Ste B",Santa Barbara,CA,93101,"restaurants, delis, sandwiches, fast food",34.430822,-119.714156,"{'Alcohol': ""u'none'"", 'Caters': 'True', 'Bike..."


## 2) Feature Preparation

In [23]:
def count_categories(cat_string):
    if pd.isna(cat_string):
        return 0
    return len([c.strip() for c in str(cat_string).split(',') if c.strip()])


def count_attributes(attr_string):
    if pd.isna(attr_string):
        return 0
    try:
        parsed = ast.literal_eval(attr_string)
        return len(parsed) if isinstance(parsed, dict) else 0
    except Exception:
        return 0


restaurant_features = df_restaurants.copy()
restaurant_features["num_categories"] = restaurant_features["categories"].apply(count_categories)
restaurant_features["num_attributes"] = restaurant_features["attributes"].apply(count_attributes)

# The restaurant export in this project does not include business-level rating/count
# columns, so compute those from the training reviews and merge them in.
business_stats = (
    df_train_reviews.groupby("business_id")["stars"]
    .agg(business_review_count="count", business_avg_stars="mean")
    .reset_index()
)

restaurant_features = restaurant_features.merge(business_stats, on="business_id", how="left")
restaurant_features["business_review_count"] = restaurant_features["business_review_count"].fillna(0)
restaurant_features["business_avg_stars"] = restaurant_features["business_avg_stars"].fillna(df_train_reviews["stars"].mean())
restaurant_features["is_open"] = 1 
restaurant_cols = [
    "business_id",
    "name",
    "city",
    "state",
    "is_open",
    "business_review_count",
    "latitude",
    "longitude",
    "business_avg_stars",
    "num_categories",
    "num_attributes",
]

restaurant_features = restaurant_features[restaurant_cols]

train_df = df_train_reviews.merge(restaurant_features, on="business_id", how="left")
test_df = df_test_reviews.merge(restaurant_features, on="business_id", how="left")

train_df = train_df.dropna(subset=["stars"]).copy()
test_df = test_df.dropna(subset=["stars"]).copy()

print("Train rows used:", train_df.shape[0])
print("Test rows used:", test_df.shape[0])


Train rows used: 41581
Test rows used: 4801


## 3) Baseline (Global Mean)

In [24]:
y_train = train_df["stars"].astype(float)
y_test = test_df["stars"].astype(float)

global_mean = y_train.mean()
baseline_pred = np.full(shape=len(y_test), fill_value=global_mean)

baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_mae = mean_absolute_error(y_test, baseline_pred)

print(f"Global mean baseline: {global_mean:.4f}")
print(f"Baseline RMSE: {baseline_rmse:.4f}")
print(f"Baseline MAE:  {baseline_mae:.4f}")


Global mean baseline: 3.9770
Baseline RMSE: 1.2493
Baseline MAE:  0.9976


## 4) Regression Pipeline

In [25]:
feature_cols = [
    "user_id",
    "business_id",
    "text",
    "city",
    "state",
    "name",
    "is_open",
    "business_review_count",
    "latitude",
    "longitude",
    "business_avg_stars",
    "num_categories",
    "num_attributes",
]

X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

# Guard against null text values before vectorization.
X_train["text"] = X_train["text"].fillna("").astype(str)
X_test["text"] = X_test["text"].fillna("").astype(str)

numeric_features = [
    "is_open",
    "business_review_count",
    "latitude",
    "longitude",
    "business_avg_stars",
    "num_categories",
    "num_attributes",
]

categorical_features = [
    "user_id",
    "business_id",
    "city",
    "state",
    "name",
]

text_feature = "text"

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

text_transformer = Pipeline(
    steps=[
        ("flatten", FunctionTransformer(lambda x: x.squeeze(), validate=False)),
        ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=3)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
        ("txt", text_transformer, [text_feature]),
    ],
    remainder="drop",
)

regressor = Ridge(alpha=2.0)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", regressor),
    ]
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

y_pred = np.clip(y_pred, 1.0, 5.0)


## 5) Evaluation

In [26]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Regression model performance")
print("-" * 40)
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R^2:  {r2:.4f}")

print("\nComparison vs baseline")
print("-" * 40)
print(f"Baseline RMSE: {baseline_rmse:.4f}")
print(f"Model RMSE:    {rmse:.4f}")
print(f"Improvement:   {baseline_rmse - rmse:.4f}")


Regression model performance
----------------------------------------
RMSE: 0.7007
MAE:  0.5072
R^2:  0.6847

Comparison vs baseline
----------------------------------------
Baseline RMSE: 1.2493
Model RMSE:    0.7007
Improvement:   0.5486


## 6) Save Predictions

In [27]:
predictions = test_df[["user_id", "business_id", "stars"]].copy()
predictions = predictions.rename(columns={"stars": "actual_stars"})
predictions["predicted_stars"] = y_pred
predictions["abs_error"] = (predictions["actual_stars"] - predictions["predicted_stars"]).abs()

predictions.to_csv("regression_predictions.csv", index=False)
predictions.head(10)


,user_id,business_id,actual_stars,predicted_stars,abs_error
0,-0EcgtUXe1rzrkmdih_tYg,hPzPpfSjgQkpWCD7YjcY-A,4.0,4.002473,0.002473
1,-1-ECBsGpG4Iw5s-ecnfqw,29YqJwOGEuAWqlHZxMc1OA,5.0,4.311667,0.688333
2,-14MA777BbjUQLw0zndvfA,Ce6B2-CYAjGvRZe3GKyxYA,2.0,1.137981,0.862019
3,-1WbN1Qd-opw8u3uEqs2Kg,mAqgsZBTN-wsShMpkz2o9g,1.0,3.125408,2.125408
4,-2hJrZCnH3vvUIET84kVJw,0qu0fNTOsSmuREYVIMPuIQ,5.0,4.711317,0.288683
5,-2ynqM2Z6pqzdUH6YXz7iQ,omnCKxL0f_akh0eYeJxjDg,4.0,3.719193,0.280807
6,-3tHJLCjgDbMNejwO9KawA,U3grYFIeu6RgAAQgdriHww,5.0,3.935397,1.064603
7,-4ATBKqwpfRewlkhmprVZQ,ssZSQ7HzLFBbm41gWubXKQ,4.0,3.734857,0.265143
8,-5JqewilyQFD_m6QlEDekg,7wSnkgOORRCqVDa_n8Upxg,5.0,4.561869,0.438131
9,-7Eh_8y1ihj3nNtdIetiRA,0RxU5OglQyPVtLKC1LPgsA,5.0,4.508011,0.491989


## 7) Example: Top Predicted Restaurants for a User (from test set candidates)

In [28]:
example_user = predictions["user_id"].iloc[0]

user_recs = (
    predictions[predictions["user_id"] == example_user]
    .sort_values("predicted_stars", ascending=False)
    .head(10)
    .merge(df_restaurants[["business_id", "name", "categories"]], on="business_id", how="left")
)

print(f"Example user_id: {example_user}")
user_recs[["business_id", "name", "predicted_stars", "actual_stars", "categories"]]


Example user_id: -0EcgtUXe1rzrkmdih_tYg


,business_id,name,predicted_stars,actual_stars,categories
0,hPzPpfSjgQkpWCD7YjcY-A,Lilac Pâtisserie,4.002473,4.0,"specialty food, health markets, restaurants, g..."
